# 🧠 ConvNets for Classification and Localization
**Author:** Sharda Vatsal Bhat  
**Dataset:** MNIST (Handwritten Digits)  
**Goal:** Build a multi-output CNN that simultaneously classifies digits and predicts bounding box coordinates.

---
### Project Overview
- Load and preprocess MNIST dataset
- Generate synthetic bounding boxes for localization
- Build a dual-head CNN (classification + localization)
- Train and evaluate the model
- Visualize predictions with bounding boxes

## Step 1: Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from tensorflow.keras.utils import to_categorical

print('TensorFlow version:', tf.__version__)
print('All libraries imported successfully!')

## Step 2: Load and Preprocess Data

In [ ]:
# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values to [0, 1] and add channel dimension
x_train = np.expand_dims(x_train / 255.0, axis=-1)  # shape: (60000, 28, 28, 1)
x_test  = np.expand_dims(x_test  / 255.0, axis=-1)  # shape: (10000, 28, 28, 1)

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test,  10)

print('Training samples :', x_train.shape)
print('Testing samples  :', x_test.shape)
print('Classes          :', 10)

## Step 3: Visualize Sample Images

In [ ]:
# Plot first 10 training samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for i in range(10):
    axes[i].imshow(x_train[i].reshape(28, 28), cmap='gray')
    axes[i].set_title(f'Label: {y_train[i]}')
    axes[i].axis('off')

plt.suptitle('Sample MNIST Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4: Generate Synthetic Bounding Boxes

In [ ]:
def generate_bboxes(n_samples):
    """
    Generate synthetic bounding boxes for localization.
    Format: [x, y, width, height] — fixed dummy values centered on the digit.
    In a real-world task, these would come from actual annotation labels.
    """
    return np.array([[5, 5, 22, 22] for _ in range(n_samples)], dtype=np.float32)

y_train_bbox = generate_bboxes(len(x_train))
y_test_bbox  = generate_bboxes(len(x_test))

print('Bounding box shape (train):', y_train_bbox.shape)
print('Sample bbox:', y_train_bbox[0])  # [x, y, width, height]

## Step 5: Build the Dual-Head CNN Model

In [ ]:
def build_model(input_shape=(28, 28, 1), num_classes=10):
    """
    Dual-head CNN:
      - Shared convolutional backbone
      - Head 1: Classification (softmax, 10 classes)
      - Head 2: Bounding box regression (linear, 4 values)
    """
    inputs = Input(shape=input_shape)

    # Shared backbone
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)

    # Head 1: Classification
    class_output = Dense(num_classes, activation='softmax', name='class_output')(x)

    # Head 2: Bounding Box Regression
    bbox_output = Dense(4, activation='linear', name='bbox_output')(x)

    model = Model(inputs=inputs, outputs=[class_output, bbox_output])
    return model


model = build_model()

model.compile(
    optimizer='adam',
    loss={
        'class_output': 'categorical_crossentropy',
        'bbox_output':  'mse'
    },
    metrics={
        'class_output': 'accuracy'
    }
)

model.summary()

## Step 6: Train the Model

In [ ]:
history = model.fit(
    x_train,
    {'class_output': y_train_cat, 'bbox_output': y_train_bbox},
    validation_data=(
        x_test,
        {'class_output': y_test_cat, 'bbox_output': y_test_bbox}
    ),
    epochs=5,
    batch_size=64
)

print('\nTraining complete!')

## Step 7: Plot Training Accuracy

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['class_output_accuracy'],     label='Train Accuracy', color='steelblue')
plt.plot(history.history['val_class_output_accuracy'], label='Val Accuracy',   color='coral')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Classification Accuracy over Epochs', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

final_acc = history.history['val_class_output_accuracy'][-1]
print(f'Final Validation Accuracy: {final_acc:.4f}')

## Step 8: Evaluate on Test Set

In [ ]:
results = model.evaluate(
    x_test,
    {'class_output': y_test_cat, 'bbox_output': y_test_bbox},
    verbose=0
)

print('Test Results:')
for name, value in zip(model.metrics_names, results):
    print(f'  {name}: {value:.4f}')

## Step 9: Visualize Predictions with Bounding Boxes

In [ ]:
def visualize_predictions(model, x_test, y_test_cat, n=10):
    """
    Predict on n test images and draw predicted bounding boxes.
    Green box = predicted bounding box
    """
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    axes = axes.flatten()

    for i in range(n):
        img = x_test[i]
        true_label = np.argmax(y_test_cat[i])

        pred_class, pred_bbox = model.predict(img.reshape(1, 28, 28, 1), verbose=0)
        pred_label = np.argmax(pred_class[0])
        bbox = pred_bbox[0]  # [x, y, w, h]

        # Convert to BGR for OpenCV drawing
        img_draw = (img * 255).astype('uint8').reshape(28, 28)
        img_bgr  = cv2.cvtColor(img_draw, cv2.COLOR_GRAY2BGR)

        # Draw bounding box
        x1, y1 = int(bbox[0]), int(bbox[1])
        x2, y2 = int(bbox[0] + bbox[2]), int(bbox[1] + bbox[3])
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 255, 0), 1)

        # Convert back to RGB for matplotlib
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        color = 'green' if pred_label == true_label else 'red'
        axes[i].imshow(img_rgb)
        axes[i].set_title(f'Pred: {pred_label} | True: {true_label}', color=color, fontsize=9)
        axes[i].axis('off')

    plt.suptitle('Predictions with Bounding Boxes (Green = Correct, Red = Wrong)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()


visualize_predictions(model, x_test, y_test_cat, n=10)

## Step 10: Final Summary

In [ ]:
final_acc = history.history['val_class_output_accuracy'][-1]

print('=' * 50)
print('   ConvNets Classification & Localization')
print('=' * 50)
print(f'Dataset          : MNIST (70,000 images)')
print(f'Input Shape      : 28 x 28 x 1 (grayscale)')
print(f'Classes          : 10 (digits 0-9)')
print(f'Architecture     : Dual-head CNN')
print(f'  Head 1         : Classification (Softmax)')
print(f'  Head 2         : Bounding Box Regression (MSE)')
print(f'Epochs           : 5')
print(f'Batch Size       : 64')
print(f'Final Val Acc    : {final_acc:.4f}')
print('=' * 50)